# logit-thermo 探索：文章の DSC その 2 ——凍らせた結晶を 1.5 まで温めて、本当に溶ける温度と溶けるときの潜熱を測る下見（Pythia-1B、GPU、L4）

- **必要な secret：なし**（`HF_TOKEN` があれば取り込みが少し速くなるだけ）。**ランタイム：L4**（ハイメモリはいりません）。見込みは 50 分〜1 時間。終わったら自動で切れます。
- もう 1 本（OLMo-2-1B）のノートブックと、別のランタイムで同時に回せます（保存先のフォルダがモデルごとに別です）。
- 探索（下見、予想なし）。どの腕も、はじめの 640 歩は同じ（0.85 → 0.35 で冷やして凍らせる、同じ種）。そこから：基本（0.35 → 1.5 を 640 歩で温める）・熟成（0.35 で 320 歩待ってから温める）・速く温める（320 歩）・強い罰（温めるあいだだけ h ＝ −1.0）の 4 本。記事 40 × サンプル 4。各歩の q（候補の確率の和）とエネルギー U も記録します。
- 計算のコード：A2-2・A2-4・A2-7・A2-7b のデータ集めのコード、`scripts/a2_explore_field.py`（`0ac42c7ab82b06d2…`）、`scripts/a2_explore_dsc.py`（`33f47a0a3bbda183…`）は Drive の前のノートブックから取り出してハッシュで確かめる。`scripts/a2_explore_dsc2.py`（`cddae78690360418…`）はこのノートブックに埋め込み。
- 保存先 `A2_explore/dsc2_pythia/`。腕は 4 本。途中で切れても、もう一度「すべて実行」すれば終わった腕は飛ばします。**結果は表示しません。**
- **通知**：🚀・腕ごとの ✅（4 回）・🎉・⚠️。名札は「logit-thermo 探索 文章の DSC その 2（1.5 まで温める・熟成・速く温める・強い罰、Pythia-1B、L4）」。終わったら、またはエラーで止まったら、ランタイムを切ります。


In [ ]:
# セル1：通知、GPU、Drive
import os, time, urllib.request
NTFY_TOPIC = 'YOUR_NTFY_TOPIC'
NB_LABEL = 'logit-thermo 探索 文章の DSC その 2（1.5 まで温める・熟成・速く温める・強い罰、Pythia-1B、L4）'
def notify(msg, tags='pig'):
    try:
        req = urllib.request.Request('https://ntfy.sh/' + NTFY_TOPIC, data=(msg + '\n[' + NB_LABEL + ']').encode('utf-8'),
                                     headers={'Title': 'logit-thermo - explore DSC2 melting Pythia-1B', 'Tags': tags})
        urllib.request.urlopen(req, timeout=10).read()
        print('通知を送りました：', msg)
    except Exception as e:
        print('通知は送れませんでした（実験には影響しません）：', type(e).__name__)
_NOTIFIED = {'error': False}
def _on_error(result):
    if result.error_in_exec is not None and not _NOTIFIED['error']:
        _NOTIFIED['error'] = True
        import traceback
        e = result.error_in_exec
        txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
        v = os.environ.get('HF_TOKEN')
        if v: txt = txt.replace(v, '***')
        txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
        notify('⚠️ 止まりました（エラー）：\n' + txt, tags='warning')
if not globals().get('_HOOK_ON'):
    get_ipython().events.register('post_run_cell', _on_error); _HOOK_ON = True
notify('🚀 スタートしました（これが届けば通知の設定は OK です）', tags='rocket')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
import torch
if not torch.cuda.is_available() or torch.cuda.get_device_properties(0).total_memory < 20e9:
    name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'GPU なし'
    notify(f'⚠️ GPU のメモリが足りません（{name}）。L4 を選んでください。ランタイムを切ります', tags='warning')
    from google.colab import runtime; runtime.unassign()
    raise SystemExit
torch.backends.cuda.matmul.allow_tf32 = False; torch.backends.cudnn.allow_tf32 = False
torch.set_float32_matmul_precision('highest')
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    pass

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# セル2：計算のコード
import ast, glob, hashlib, json
def take_src(want):
    """want：{変数の名前: sha256}。Drive の logit-thermo のノートブックから、埋め込んだコードを取り出してハッシュで確かめる。"""
    got = {}
    for p in sorted(glob.glob('/content/drive/MyDrive/logit-thermo' + '/**/*.ipynb', recursive=True)):
        try:
            cells = json.load(open(p))['cells']
        except Exception:
            continue
        for c in cells:
            s = c['source'] if isinstance(c['source'], str) else ''.join(c['source'])
            if c.get('cell_type') != 'code' or 'SRC' not in s:
                continue
            try:
                body = ast.parse(s).body
            except SyntaxError:
                continue
            for n in body:
                if isinstance(n, ast.Assign) and isinstance(n.targets[0], ast.Name) and n.targets[0].id in want and n.targets[0].id not in got:
                    v = ast.literal_eval(n.value)
                    if hashlib.sha256(v.encode()).hexdigest() == want[n.targets[0].id]:
                        got[n.targets[0].id] = v
        if len(got) == len(want):
            break
    miss = [k for k in want if k not in got]
    assert not miss, ('Drive のノートブックからコードを取り出せませんでした（前のノートブックを動かしていませんか）：', miss)
    return got
_s = take_src({'SRC22': 'e578cfc480980be6b0d1e18377b6beb8b7555e0744f56b13f63cb304a9f73d53', 'SRC24': '6ef87866ae7d895d63c6e07d48a58a2ddf4111a4802ffe98dbe7f61892c96b57', 'SRC7C': '8f8776cbe4a0e1a7b1f676c4f84ba70fb9c6c0b46ef1466969e09f0ba062b550', 'SRC7BC': 'b03ff9e2b16e2297fb9142a26e15f561caacf77216453d78e2024066c01c57d0', 'SRCF': '0ac42c7ab82b06d2e7443ba9c0c0e147c0121b5351e0b5830b1a54b46043190d', 'SRCD': '33f47a0a3bbda18341205dedb94c6d200f295fc4136f5cb8e0ee067b2d7f518c'})
SRC22, SRC24, SRC7C, SRC7BC, SRCF, SRCD = _s['SRC22'], _s['SRC24'], _s['SRC7C'], _s['SRC7BC'], _s['SRCF'], _s['SRCD']
SRCD2 = '"""探索 8.38（下見）：文章の DSC その 2 ——凍らせた結晶を 1.5 まで温めて、本当に溶ける温度と溶けるときの潜熱を測る（2026-10-31 16:5x、GPU（L4）、判定しない）。\nscripts/a2_2_collect.py・a2_4_collect.py・a2_7_collect.py・a2_7b_collect.py・a2_explore_field.py・a2_explore_dsc.py を実行したあとの同じ名前空間で実行する\n（load_model・valid_mask・split_doc・CopyIndexN・B7_N・generate_field・generate_dsc・D_MODELS・D_CHUNK を変えずに使う）。\n\nどの腕も、はじめの 640 歩は同じ（0.85 → 0.35 で冷やす、場なし、同じ種）。なので、どの腕も「同じように凍った結晶」から温めはじめる（対になった比べ）。\n腕：\n  \'base\'     ：冷やす 640 歩 → 温める 0.35 → 1.5 を 640 歩（場なし）\n  \'hold\'     ：冷やす 640 歩 → 0.35 で 320 歩待つ（熟成）→ 温める 640 歩（場なし）\n  \'fastheat\' ：冷やす 640 歩 → 温める 0.35 → 1.5 を 320 歩（場なし）\n  \'penheat\'  ：冷やす 640 歩（場なし）→ 温める 640 歩のあいだだけ h ＝ −1.0（強いくり返しの罰）\n記録：トークン、候補があったか、候補を選んだか、U ＝ −ln p（温度 1・場なし、選んだトークン）、q0（場をかける前の、その温度での候補の確率の和）、\n  q（場をかけた後の候補の確率の和）、歩ごとの温度と場。確率の計算は倍精度、保存は float32。\n"""\nimport gc, os, time\nimport numpy as np\nimport torch\n\nD2_COOL = (0.85, 0.35, 640)\nD2_HEAT_TOP = 1.5\nD2_ARMS = [\'base\', \'hold\', \'fastheat\', \'penheat\']\nD2_S = 4\nD2_SEED = 20261137\nD2_PEN = -1.0\n\n\ndef d2_sched(arm):\n    """→ (温度の列, 場の列)。"""\n    a, b, n = D2_COOL\n    cool = np.linspace(a, b, n)\n    heat = np.linspace(b, D2_HEAT_TOP, 320 if arm == \'fastheat\' else 640)\n    parts = [cool] + ([np.full(320, b)] if arm == \'hold\' else []) + [heat]\n    Ts = np.concatenate(parts)\n    Hs = np.zeros_like(Ts)\n    if arm == \'penheat\':\n        Hs[n:] = D2_PEN\n    return Ts, Hs\n\n\ndef d2_seed(s, c):\n    return D2_SEED + 10 * s + c\n\n\n@torch.no_grad()\ndef generate_dsc2(model, prompts, Ts, Hs, valid, gen):\n    """温度と場が歩ごとに変わる。手順と乱数の使い方は generate_field・generate_dsc と同じ。"""\n    device = valid.device\n    B = len(prompts)\n    x = torch.tensor(prompts, device=device)\n    out = model(x, use_cache=True)\n    past, logits = out.past_key_values, out.logits[:, -1, :].float()\n    idx = [CopyIndexN(p, B7_N) for p in prompts]\n    rec = {k: [] for k in (\'toks\', \'cand\', \'pick\', \'U\', \'q0\', \'q\')}\n    for t in range(len(Ts)):\n        T, h = float(Ts[t]), float(Hs[t])\n        z = logits.double().masked_fill(~valid, float(\'-inf\'))\n        lp1 = torch.log_softmax(z, dim=-1)\n        cands = [idx[b].candidates() for b in range(B)]\n        has = torch.tensor([len(c) > 0 for c in cands], device=device)\n        rows, cols = [], []\n        for b, c in enumerate(cands):\n            rows += [b] * len(c); cols += list(c)\n        rt = torch.tensor(rows, device=device, dtype=torch.long); ct = torch.tensor(cols, device=device, dtype=torch.long)\n        q0 = torch.zeros(B, dtype=torch.float64, device=device)\n        if rows:\n            q0.index_add_(0, rt, torch.softmax(z / T, dim=-1)[rt, ct])\n        if h != 0.0 and rows:\n            z[rt, ct] += h\n        p = torch.softmax(z / T, dim=-1)\n        q = torch.zeros(B, dtype=torch.float64, device=device)\n        if rows:\n            q.index_add_(0, rt, p[rt, ct])\n        nxt = torch.multinomial(p, 1, generator=gen)\n        pk = torch.tensor([int(nxt[b, 0]) in set(cands[b]) for b in range(B)], device=device)\n        for b in range(B):\n            idx[b].push(int(nxt[b, 0]))\n        for k, v in ((\'toks\', nxt[:, 0]), (\'cand\', has), (\'pick\', pk), (\'U\', -lp1.gather(-1, nxt)[:, 0]), (\'q0\', q0), (\'q\', q)):\n            rec[k].append(v)\n        out = model(nxt, past_key_values=past, use_cache=True)\n        past, logits = out.past_key_values, out.logits[:, -1, :].float()\n    st = lambda v: torch.stack(v, 1).cpu().numpy()\n    dt = dict(toks=np.int32, cand=np.bool_, pick=np.bool_, U=np.float32, q0=np.float32, q=np.float32)\n    return {k: st(v).astype(dt[k]) for k, v in rec.items()}\n\n\ndef run_dsc2(name, hf_id, docs, outdir, device, notify=print):\n    todo = [a for a in D2_ARMS if not os.path.exists(f\'{outdir}/{name}@{a}.npz\')]\n    if not todo:\n        print(name, \'はもう全部終わっています（飛ばします）\')\n        return\n    t0 = time.time()\n    tok, model = load_model(hf_id, device)\n    assert next(model.parameters()).dtype == torch.float32\n    valid, eos = valid_mask(tok, model, device)\n    pairs = [split_doc(tok, d[\'text\']) for d in docs]\n    use = [i for i, p in enumerate(pairs) if p is not None]\n    nD = len(use)\n    P = {len(pairs[i][0]) for i in use}\n    assert len(P) == 1\n    prompts = [pairs[i][0] for i in use]\n    maxpos = getattr(model.config, \'max_position_embeddings\', None)\n    assert maxpos is None or P.pop() + max(len(d2_sched(a)[0]) for a in D2_ARMS) <= maxpos, (\'文が長すぎます\', maxpos)\n    chunks = [list(range(i, min(nD, i + D_CHUNK))) for i in range(0, nD, D_CHUNK)]\n    n_cool = D2_COOL[2]\n    cool_ref = None\n    for arm in todo:\n        ta = time.time()\n        Ts, Hs = d2_sched(arm)\n        L = len(Ts)\n        dt = dict(toks=np.int32, cand=np.bool_, pick=np.bool_, U=np.float32, q0=np.float32, q=np.float32)\n        R = {k: np.zeros((D2_S, nD, L), v) for k, v in dt.items()}\n        for s in range(D2_S):\n            for c, ii in enumerate(chunks):\n                g = torch.Generator(device=device).manual_seed(d2_seed(s, c))\n                r = generate_dsc2(model, [prompts[i] for i in ii], Ts, Hs, valid, g)\n                for k in R:\n                    R[k][s, ii] = r[k]\n                torch.cuda.empty_cache()\n        if cool_ref is None:\n            cool_ref = R[\'toks\'][:, :, :n_cool].copy()\n        else:\n            assert (R[\'toks\'][:, :, :n_cool] == cool_ref).all(), (name, arm, \'はじめの冷やす 640 歩が、ほかの腕と違います\')\n        path = f\'{outdir}/{name}@{arm}.npz\'\n        np.savez_compressed(path + \'.part.npz\', Ts=Ts, Hs=Hs, arm=arm, use=np.array(use), seconds=time.time() - ta, chunk=D_CHUNK, **R)\n        os.replace(path + \'.part.npz\', path)\n        notify(f\'{name}@{arm} おわり（{time.time() - ta:.0f} 秒、通算 {time.time() - t0:.0f} 秒）\')\n    del model; gc.collect(); torch.cuda.empty_cache()\n\n\n@torch.no_grad()\ndef self_test_dsc2(device):\n    """小さなランダムのモデルで：(1) 温度・場が一定なら generate_field と同じトークン。(2) 場が 0 なら generate_dsc と同じトークンと U。\n    (3) q を読み直しで確かめる。(4) 腕の予定（長さ・温度・場）。"""\n    from transformers import AutoModelForCausalLM, AutoTokenizer\n    hf_id = \'hf-internal-testing/tiny-random-GPT2LMHeadModel\'\n    tok = AutoTokenizer.from_pretrained(hf_id)\n    model = AutoModelForCausalLM.from_pretrained(hf_id, torch_dtype=torch.float32).to(device).eval()\n    valid, _ = valid_mask(tok, model, device)\n    global CONT\n    keep = CONT\n    CONT = 60\n    try:\n        prompts = [[5, 6, 7, 8] * 4, [9, 10, 11, 12] * 4, [3, 4, 3, 4] * 4]\n        for h in (0.0, 0.3):\n            a = generate_field(model, prompts, 0.7, valid, torch.Generator(device=device).manual_seed(5), h)\n            b = generate_dsc2(model, prompts, np.full(60, 0.7), np.full(60, h), valid, torch.Generator(device=device).manual_seed(5))\n            assert (a[\'toks\'] == b[\'toks\']).all() and (a[\'pick\'] == b[\'pick\']).all(), (\'generate_field と違う\', h)\n        Ts = np.linspace(0.9, 0.4, 60)\n        a = generate_dsc(model, prompts, Ts, 0.0, valid, torch.Generator(device=device).manual_seed(6))\n        b = generate_dsc2(model, prompts, Ts, np.zeros(60), valid, torch.Generator(device=device).manual_seed(6))\n        assert (a[\'toks\'] == b[\'toks\']).all() and np.allclose(a[\'U\'], b[\'U\']), \'generate_dsc と違う\'\n        assert np.array_equal(b[\'q\'], b[\'q0\'])\n        # 場が途中から入る：q ≥ q0（候補がある歩）、読み直しの q と合う\n        Hs = np.r_[np.zeros(30), np.full(30, -1.0)]\n        b = generate_dsc2(model, prompts, Ts, Hs, valid, torch.Generator(device=device).manual_seed(7))\n        assert (b[\'q\'][:, 30:][b[\'cand\'][:, 30:]] <= b[\'q0\'][:, 30:][b[\'cand\'][:, 30:]] + 1e-6).all()\n        seq = b[\'toks\'][0].tolist()\n        x = torch.tensor([prompts[0] + seq], device=device)\n        lg = model(x).logits[0, len(prompts[0]) - 1: len(prompts[0]) - 1 + 60].double().masked_fill(~valid, float(\'-inf\'))\n        ci = CopyIndexN(prompts[0], B7_N)\n        for t in range(60):\n            c = list(ci.candidates())\n            zz = lg[t].clone()\n            if c and Hs[t] != 0.0:\n                zz[c] += Hs[t]\n            pp = torch.softmax(zz / Ts[t], -1)\n            qq = float(pp[c].sum()) if c else 0.0\n            assert abs(qq - float(b[\'q\'][0, t])) < 2e-3, (t, qq, float(b[\'q\'][0, t]))\n            ci.push(seq[t])\n    finally:\n        CONT = keep\n    for arm in D2_ARMS:\n        Ts, Hs = d2_sched(arm)\n        assert abs(Ts[0] - 0.85) < 1e-12 and abs(Ts[639] - 0.35) < 1e-12 and abs(Ts[-1] - D2_HEAT_TOP) < 1e-12\n        assert (Hs[:640] == 0).all() and ((Hs[640:] == D2_PEN).all() if arm == \'penheat\' else (Hs == 0).all())\n    assert [len(d2_sched(a)[0]) for a in D2_ARMS] == [1280, 1600, 960, 1280]\n    return \'OK\'\n'
assert hashlib.sha256(SRCD2.encode()).hexdigest() == 'cddae786903604183e46708a6606b482f720bba0b0df0a8df724440c9bde51b2'
exec(SRC22)
exec(SRC24)
exec(SRC7C)
exec(SRC7BC)
exec(SRCF)
exec(SRCD)
exec(SRCD2)
print('コードを置きました')


In [ ]:
# セル3：自己テスト → 記事（A2-2 と同じか確かめる）→ 4 本の腕 → 保存（結果は表示しない）
import json, sys, transformers
try:
    print('自己テスト：', self_test_a27('cuda'), self_test_7b('cuda'), self_test_field('cuda'), self_test_dsc('cuda'), self_test_dsc2('cuda'))
    os.makedirs('/content/drive/MyDrive/logit-thermo/A2_explore/dsc2_pythia', exist_ok=True)
    assert SEED == 20261027, ('記事を選ぶ種が A2-2 と違います', SEED)
    docs, meta = load_docs()
    old = json.load(open('/content/drive/MyDrive/logit-thermo' + '/A2_2/docs_meta.json'))
    assert [d['text_sha256'] for d in old['docs']] == [d['text_sha256'] for d in meta['docs']], 'BBC の記事が A2-2 と違います'
    name, hf_id = D_MODELS['pythia']
    run_dsc2(name, hf_id, docs, '/content/drive/MyDrive/logit-thermo/A2_explore/dsc2_pythia', 'cuda', notify=lambda s: notify('✅ ' + s, tags='white_check_mark'))
    json.dump(dict(model=[name, hf_id], arms=D2_ARMS, cool=D2_COOL, heat_top=D2_HEAT_TOP, pen=D2_PEN, samples=D2_S, chunk=D_CHUNK, seed=D2_SEED,
                   dsc2_sha256='cddae786903604183e46708a6606b482f720bba0b0df0a8df724440c9bde51b2', torch=torch.__version__, transformers=transformers.__version__,
                   python=sys.version.split()[0], gpu=torch.cuda.get_device_name(0), matmul=torch.get_float32_matmul_precision()),
              open('/content/drive/MyDrive/logit-thermo/A2_explore/dsc2_pythia' + f'/run_meta_{name}.json', 'w'), ensure_ascii=False, indent=1)
    print('保存しました')
except Exception as e:
    import traceback
    _NOTIFIED['error'] = True
    txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
    v = os.environ.get('HF_TOKEN')
    if v: txt = txt.replace(v, '***')
    txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
    notify('⚠️ 探索 文章の DSC その 2（Pythia-1B）で止まりました。ランタイムを切ります：\n' + txt, tags='warning')
    if globals().get('AUTO_DISCONNECT', True):
        from google.colab import drive as _d, runtime as _r
        _d.flush_and_unmount(); time.sleep(3); _r.unassign()
    raise


In [ ]:
# 最後：終わりの通知と、ランタイムを切る
notify('🎉 終わりました：文章の DSC その 2（Pythia-1B）の材料を A2_explore/dsc2_pythia に保存しました', tags='tada')
if globals().get('AUTO_DISCONNECT', True):
    from google.colab import drive as _drive, runtime as _runtime
    _drive.flush_and_unmount(); time.sleep(3); _runtime.unassign()
